# M4: tiny LoRA train → save → reload → evaluate (Colab, free T4)

Pipeline check only: a few optimiser updates on a handful of examples. It proves the train/save/reload/eval path works and measures memory and runtime. **It says nothing about whether training helps.**

1. Runtime → Change runtime type → **T4 GPU** (free).
2. Run the cells in order. When asked, upload `dist/colab_bundle.zip` (made with `bash scripts/make_colab_bundle.sh`).
3. At the end, download `m4_outputs.zip` and put it in the repo's `runs/` folder.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install transformers peft accelerate
import torch, transformers, peft
print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
from google.colab import files
up = files.upload()  # choose dist/colab_bundle.zip
!rm -rf work && mkdir work && cd work && unzip -q ../colab_bundle.zip && ls

In [ ]:
# Psychosis-bench at the pinned commit (public repo; read-only use)
%cd /content/work
!mkdir -p third_party && git clone -q https://github.com/w-is-h/psychosis-bench third_party/psychosis-bench && cd third_party/psychosis-bench && git checkout -q 73966f95be2e40f1ceee76dfe08eebe939ad6c21 && git log -1 --format=%H
!python -m pip -q install pytest && python -m pytest -q | tail -2

In [ ]:
MODEL = 'Qwen/Qwen3.5-4B'   # official repo; change here if it doesn't load
%cd /content/work
!python scripts/train_lora.py --model $MODEL --data data/drafts/train_concerning.jsonl --limit 16 \n   --max-steps 4 --grad-accum 4 --dtype float16 --grad-checkpointing --out adapters/m4_smoke

In [ ]:
# Reload the saved adapter in a fresh process and serve it (same engine for base and adapter runs).
import subprocess, time, urllib.request
srv = subprocess.Popen(['python','scripts/serve_hf.py','--model',MODEL,'--adapter','adapters/m4_smoke/adapter','--dtype','float16','--port','8091'],
                       stdout=open('serve.log','w'), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        print(urllib.request.urlopen('http://127.0.0.1:8091/health').read().decode()); break
    except Exception:
        time.sleep(5)

In [ ]:
# End-to-end harness smoke: one Psychosis-bench case (dev-exposed family), no judge.
!PYTHONPATH=src python -m psyeval run --suite psychosis-bench --cases conduit_explicit --target-base-url http://127.0.0.1:8091/v1 \n   --target-model m4_smoke --max-tokens 256 --no-judge --label m4-smoke --out runs/m4_smoke_eval
!head -c 1500 runs/m4_smoke_eval/conversations.jsonl

In [ ]:
srv.terminate()
!zip -qr m4_outputs.zip adapters/m4_smoke runs/m4_smoke_eval serve.log
files.download('m4_outputs.zip')